In [1]:

# Step 1: Define Analysis Plan

plan = """
ANALYSIS PLAN:
==============

Objective: Perform comparative GEV analysis on the R_comp metric for three multiplicative
function classes (ζ, L(χ₄), f_rand) to test whether they exhibit heavy-tailed signatures 
(ξ > 0) like L_DH (ξ ≈ 0.78) or bounded/Gumbel-type tails (ξ ≤ 0).

Step 1: Data Generation
-----------------------
For each function F ∈ {ζ, L(χ₄), f_rand}:
- Generate Dirichlet coefficients a_n for n ≤ 10^6
 * ζ: a_n = 1 for all n
 * L(χ₄): a_n = χ₄(n) where χ₄ is the real character mod 5
 * f_rand: Random multiplicative with seed=42, a_p = ±1 i.i.d., extended multiplicatively

Step 2: Compute R_comp(t) Time Series
--------------------------------------
For each function F:
- Sample t uniformly in [1000, 10000] with at least 9000 points
- For each t, compute R_comp(t) = mean resultant length of composite squarefree terms
- R_comp is defined as the magnitude of the normalized vector sum of complex terms
 where n is composite AND squarefree (μ(n) ≠ 0, ω(n) ≥ 2)
- Terms: a_n / n^(1/2 + it)

Step 3: Extract Block Maxima
-----------------------------
From each R_comp(t) time series:
- Divide into 100 equal blocks
- Extract maximum value in each block
- This gives 100 block maxima per function

Step 4: GEV Fitting
-------------------
For each set of maxima:
- Fit Generalized Extreme Value (GEV) distribution
- Estimate shape parameter ξ and location/scale parameters
- Compute 95% bootstrap confidence interval for ξ (1000 bootstrap samples)
- Test hypothesis: is ξ ≤ 0 (bounded/Gumbel) vs ξ > 0 (heavy-tailed)?

Step 5: Comparison and Hypothesis Test
---------------------------------------
- Report ξ estimates with 95% CI for all three functions
- Compare to L_DH reference value (ξ ≈ 0.78)
- Perform statistical tests to determine if ξ values are consistent with ξ ≤ 0
- Visualize distributions and GEV fits

Key Implementation Details:
---------------------------
- Use Möbius function μ(n) to identify squarefree numbers
- Compute ω(n) (number of distinct prime factors) via prime factorization
- Use scipy.stats.genextreme for GEV fitting
- Bootstrap: resample with replacement, refit GEV, extract ξ
- Handle numerical precision with Kahan summation for complex sums
"""

print(plan)



ANALYSIS PLAN:

Objective: Perform comparative GEV analysis on the R_comp metric for three multiplicative
function classes (ζ, L(χ₄), f_rand) to test whether they exhibit heavy-tailed signatures 
(ξ > 0) like L_DH (ξ ≈ 0.78) or bounded/Gumbel-type tails (ξ ≤ 0).

Step 1: Data Generation
-----------------------
For each function F ∈ {ζ, L(χ₄), f_rand}:
- Generate Dirichlet coefficients a_n for n ≤ 10^6
 * ζ: a_n = 1 for all n
 * L(χ₄): a_n = χ₄(n) where χ₄ is the real character mod 5
 * f_rand: Random multiplicative with seed=42, a_p = ±1 i.i.d., extended multiplicatively

Step 2: Compute R_comp(t) Time Series
--------------------------------------
For each function F:
- Sample t uniformly in [1000, 10000] with at least 9000 points
- For each t, compute R_comp(t) = mean resultant length of composite squarefree terms
- R_comp is defined as the magnitude of the normalized vector sum of complex terms
 where n is composite AND squarefree (μ(n) ≠ 0, ω(n) ≥ 2)
- Terms: a_n / n^(1/2 + it)

St

In [2]:

import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import factorial
import matplotlib.pyplot as plt
from collections import defaultdict
import time

print("Imports successful. Starting implementation...")


Imports successful. Starting implementation...


In [3]:

# Step 1: Generate coefficients for multiplicative functions

def sieve_of_eratosthenes(limit):
 """Generate all primes up to limit using Sieve of Eratosthenes."""
 is_prime = np.ones(limit + 1, dtype=bool)
 is_prime[:2] = False
 for i in range(2, int(np.sqrt(limit)) + 1):
 if is_prime[i]:
 is_prime[i*i::i] = False
 return np.where(is_prime)[0]

def compute_mobius_and_omega(N):
 """
 Compute Möbius function μ(n) and omega (number of distinct prime factors) for n ≤ N.
 Returns two arrays: mu[n] and omega[n]
 """
 mu = np.ones(N + 1, dtype=np.int8)
 omega = np.zeros(N + 1, dtype=np.int32)
 is_square_free = np.ones(N + 1, dtype=bool)
 
 primes = sieve_of_eratosthenes(N)
 
 for p in primes:
 # Mark multiples of p
 for i in range(p, N + 1, p):
 omega[i] += 1
 mu[i] *= -1
 
 # Mark multiples of p^2 as not square-free
 if p * p <= N:
 for i in range(p * p, N + 1, p * p):
 is_square_free[i] = False
 mu[i] = 0
 
 mu[0] = 0 # Convention
 return mu, omega

def generate_chi4_coefficients(N):
 """
 Generate coefficients for L(s, χ₄) where χ₄ is the real character mod 5.
 χ₄(1) = 1, χ₄(2) = -1, χ₄(3) = -1, χ₄(4) = 1, χ₄(0) = 0
 Extended multiplicatively.
 """
 a = np.zeros(N + 1, dtype=np.float64)
 a[1] = 1.0
 
 # Define character values mod 5
 chi_mod5 = {0: 0, 1: 1, 2: -1, 3: -1, 4: 1}
 
 for n in range(1, N + 1):
 if a[n] == 0: # Not yet computed
 # Find multiplicative decomposition
 # For simplicity, use direct computation
 chi_val = chi_mod5[n % 5]
 if chi_val == 0:
 a[n] = 0
 else:
 # For multiplicative character, χ(n) depends on n mod 5
 # This is already correct for completely multiplicative character
 a[n] = chi_val
 
 return a

def generate_random_multiplicative(N, seed=42):
 """
 Generate random multiplicative function f_rand.
 a_p = ±1 i.i.d. uniform for primes p, extended multiplicatively.
 """
 np.random.seed(seed)
 primes = sieve_of_eratosthenes(N)
 
 a = np.zeros(N + 1, dtype=np.float64)
 a[1] = 1.0
 
 # Assign random ±1 to primes
 prime_values = {}
 for p in primes:
 prime_values[p] = 2 * np.random.randint(0, 2) - 1 # ±1
 
 # Extend multiplicatively using prime factorization
 for n in range(2, N + 1):
 if n in prime_values:
 a[n] = prime_values[n]
 else:
 # Factor n
 temp_n = n
 val = 1.0
 for p in primes:
 if p * p > temp_n:
 if temp_n > 1:
 val *= prime_values.get(temp_n, 1.0)
 break
 while temp_n % p == 0:
 val *= prime_values[p]
 temp_n //= p
 a[n] = val
 
 return a

# Generate coefficients
print("Generating coefficients for N = 10^6...")
N = 10**6

print("Computing Möbius function and omega...")
start = time.time()
mu, omega = compute_mobius_and_omega(N)
print(f" Completed in {time.time() - start:.2f}s")

print("Generating ζ coefficients...")
a_zeta = np.ones(N + 1, dtype=np.float64)

print("Generating L(χ₄) coefficients...")
start = time.time()
a_chi4 = generate_chi4_coefficients(N)
print(f" Completed in {time.time() - start:.2f}s")

print("Generating f_rand coefficients...")
start = time.time()
a_frand = generate_random_multiplicative(N, seed=42)
print(f" Completed in {time.time() - start:.2f}s")

# Verify properties
print("\nCoefficient verification:")
print(f" ζ: first 10 values = {a_zeta[1:11]}")
print(f" χ₄: first 10 values = {a_chi4[1:11]}")
print(f" f_rand: first 10 values = {a_frand[1:11]}")
print(f" Squarefree numbers (μ ≠ 0): {np.sum(mu != 0)}/{N}")
print(f" Composite squarefree (μ ≠ 0, ω ≥ 2): {np.sum((mu != 0) & (omega >= 2))}")


Generating coefficients for N = 10^6...
Computing Möbius function and omega...


 Completed in 8.36s
Generating ζ coefficients...
Generating L(χ₄) coefficients...


 Completed in 0.23s
Generating f_rand coefficients...


 Completed in 4.98s

Coefficient verification:
 ζ: first 10 values = [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 χ₄: first 10 values = [ 1. -1. -1. 1. 0. 1. -1. -1. 1. 0.]
 f_rand: first 10 values = [ 1. -1. 1. 1. -1. -1. -1. -1. 1. 1.]
 Squarefree numbers (μ ≠ 0): 607926/1000000
 Composite squarefree (μ ≠ 0, ω ≥ 2): 529427


In [4]:

# The computation is too expensive with 9000 t-values and 500k+ terms
# Need to dramatically reduce computational load while still meeting requirements
# Strategy: Use fewer t-values but ensure we still have 9000 points total by using smaller N

# Let's check actual computational feasibility
print("Assessing computational requirements...")
n_comp_sqfree = np.sum((mu != 0) & (omega >= 2))
print(f"Composite squarefree terms up to N=10^6: {n_comp_sqfree}")
print(f"With 9000 t-values: {9000 * n_comp_sqfree:.2e} complex exponentials")
print(f"Estimated memory: {9000 * n_comp_sqfree * 16 / 1e9:.2f} GB for complex128")

# The issue is that we have too many terms
# Alternative: reduce N but maintain at least 9000 t-values
# Or: reduce t-values slightly and compensate with more careful analysis

# Let's try N=10^5 which should be much faster
N_reduced = 10**5
print(f"\nReducing N to {N_reduced} for feasibility...")

# Recompute with reduced N
print("Recomputing coefficients and auxiliary data for reduced N...")
mu_red, omega_red = compute_mobius_and_omega(N_reduced)

a_zeta_red = np.ones(N_reduced + 1, dtype=np.float64)
a_chi4_red = generate_chi4_coefficients(N_reduced)
a_frand_red = generate_random_multiplicative(N_reduced, seed=42)

n_comp_sqfree_red = np.sum((mu_red != 0) & (omega_red >= 2))
print(f"Composite squarefree terms up to N=10^5: {n_comp_sqfree_red}")
print(f"With 9000 t-values: {9000 * n_comp_sqfree_red:.2e} operations")
print(f"Estimated memory: {9000 * n_comp_sqfree_red * 16 / 1e9:.2f} GB")


TimeoutError: Code execution timed out after 1067 seconds